# **BLOCK 1: INSTALL YOLOv8 AND DEPENDENCIES**

In [ ]:
# ============================================================================
# BLOCK 1: INSTALL YOLOv8 AND DEPENDENCIES
# ============================================================================

print("="*70)
print("INSTALLING YOLOv8 AND DEPENDENCIES")
print("="*70)

# Install ultralytics
!pip install ultralytics -q

# Verify installation
import ultralytics
from ultralytics import YOLO
import torch
import os
import yaml
import shutil
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

print(f"✅ YOLOv8 version: {ultralytics.__version__}")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("="*70)

# **BLOCK 2: DATASET PATHS AND VERIFICATION**

In [ ]:
# ============================================================================
# BLOCK 2: DATASET PATHS AND VERIFICATION
# ============================================================================

print("="*70)
print("DATASET VERIFICATION")
print("="*70)

# Your dataset path on Kaggle
BASE_PATH = "/kaggle/input/datasets/andrewwageh111/arm-and-shoulder-fracture-dataset/Filtered_Shoulder_Arm_Dataset_FINAL/Filtered_Shoulder_Arm_Dataset_FINAL"

# Paths to splits
TRAIN_IMAGES = os.path.join(BASE_PATH, "train", "images")
TRAIN_LABELS = os.path.join(BASE_PATH, "train", "labels")
VAL_IMAGES = os.path.join(BASE_PATH, "val", "images")
VAL_LABELS = os.path.join(BASE_PATH, "val", "labels")
TEST_IMAGES = os.path.join(BASE_PATH, "test", "images")
TEST_LABELS = os.path.join(BASE_PATH, "test", "labels")

WORKING_DIR = "/kaggle/working"

print(f"📁 Base path: {BASE_PATH}")
print(f"📁 Train images: {TRAIN_IMAGES}")
print(f"📁 Train labels: {TRAIN_LABELS}")
print(f"📁 Val images: {VAL_IMAGES}")
print(f"📁 Test images: {TEST_IMAGES}")

# Verify paths exist
for path, name in [(TRAIN_IMAGES, "Train images"), (TRAIN_LABELS, "Train labels"),
                   (VAL_IMAGES, "Val images"), (VAL_LABELS, "Val labels"),
                   (TEST_IMAGES, "Test images"), (TEST_LABELS, "Test labels")]:
    exists = os.path.exists(path)
    print(f"  {name}: {'✅' if exists else '❌'}")

# Count images
train_count = len([f for f in os.listdir(TRAIN_IMAGES) if f.endswith(('.jpg', '.jpeg', '.png'))])
val_count = len([f for f in os.listdir(VAL_IMAGES) if f.endswith(('.jpg', '.jpeg', '.png'))])
test_count = len([f for f in os.listdir(TEST_IMAGES) if f.endswith(('.jpg', '.jpeg', '.png'))])

print(f"\n📊 DATASET STATISTICS:")
print(f"   Training: {train_count} images")
print(f"   Validation: {val_count} images")
print(f"   Test: {test_count} images")
print(f"   TOTAL: {train_count + val_count + test_count} images")
print("="*70)

# **BLOCK 3: CREATE DATA.YAML CONFIGURATION FILE**

In [ ]:
# ============================================================================
# BLOCK 3: CREATE DATA.YAML CONFIGURATION FILE
# ============================================================================

print("="*70)
print("CREATING DATA.YAML FOR YOLOv8")
print("="*70)

# Create data.yaml for YOLOv8
data_yaml = {
    'path': BASE_PATH,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 1,  # number of classes
    'names': ['fracture']
}

# Save to working directory
yaml_path = os.path.join(WORKING_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"✅ data.yaml saved to: {yaml_path}")
print("\n📋 data.yaml contents:")
print("="*40)
with open(yaml_path, 'r') as f:
    print(f.read())
print("="*40)

# **BLOCK 4: VALIDATE YOLO LABEL FORMAT**

In [ ]:
# ============================================================================
# BLOCK 4: VALIDATE YOLO LABEL FORMAT
# ============================================================================

print("="*70)
print("VALIDATING YOLO LABEL FORMAT")
print("="*70)

def validate_yolo_labels(images_dir, labels_dir, max_samples=5):
    """Check if label format is correct"""
    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    valid_count = 0
    invalid_count = 0
    
    print(f"\n📋 Checking first {max_samples} images...")
    
    for img_file in image_files[:max_samples]:
        label_file = os.path.splitext(img_file)[0] + '.txt'
        label_path = os.path.join(labels_dir, label_file)
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            print(f"\n   Image: {img_file}")
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = parts[0]
                    x_center, y_center, width, height = parts[1:5]
                    print(f"      Class: {class_id}, bbox: [{x_center}, {y_center}, {width}, {height}]")
                    valid_count += 1
                else:
                    print(f"      ❌ Invalid format: {line.strip()}")
                    invalid_count += 1
        else:
            print(f"   ❌ No label file for: {img_file}")
            invalid_count += 1
    
    print(f"\n✅ YOLO format validation complete!")
    return valid_count, invalid_count

valid, invalid = validate_yolo_labels(TRAIN_IMAGES, TRAIN_LABELS, max_samples=5)
print("="*70)

# **BLOCK 5: TRAIN YOLOv8 ON SHOULDER/ARM FRACTURE DATASET**

In [ ]:
# ============================================================================
# BLOCK 5: TRAIN YOLOv8 ON SHOULDER/ARM FRACTURE DATASET
# ============================================================================

print("="*70)
print("TRAINING YOLOv8 ON SHOULDER/ARM FRACTURE DATASET")
print("="*70)
print(f"📊 Training images: {train_count}")
print(f"🎯 Target epochs: 100")
print(f"⏱️ Estimated time: 12-15 hours")
print(f"💾 Batch size: 16")
print(f"🖼️ Image size: 640")
print("="*70)

# Initialize YOLOv8 with pretrained weights
model = YOLO('yolov8m.pt')  # Medium model - best balance

# Training parameters
results = model.train(
    data=yaml_path,
    epochs=100,              # 100 epochs as recommended
    imgsz=640,               # Standard YOLO input size
    batch=16,                # Works well with T4 16GB GPU
    device=0,                # Use GPU (0 = first GPU)
    workers=8,               # Data loading workers
    
    # Save settings
    save=True,               # Save checkpoints
    save_period=10,          # Save every 10 epochs
    project='shoulder_arm_yolo',
    name='run1_100epochs',
    exist_ok=True,
    
    # Early stopping
    patience=20,             # Stop if no improvement for 20 epochs
    
    # Optimizer settings
    lr0=0.01,                # Initial learning rate
    lrf=0.01,                # Final learning rate factor
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    
    # Data augmentation (beneficial for medical imaging)
    mosaic=1.0,              # Mosaic augmentation
    mixup=0.1,               # Mixup augmentation
    copy_paste=0.1,          # Copy-paste augmentation
    hsv_h=0.015,             # Hue augmentation
    hsv_s=0.7,               # Saturation augmentation
    hsv_v=0.4,               # Value augmentation
    degrees=0.0,             # Rotation (0 for medical images)
    translate=0.1,           # Translation
    scale=0.5,               # Scaling
    shear=0.0,               # Shear
    perspective=0.0,         # Perspective
    flipud=0.0,              # Flip up-down (usually 0 for X-rays)
    fliplr=0.5,              # Flip left-right
)

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)

# **BLOCK 6: EVALUATE ON VALIDATION SET**

In [ ]:
# ============================================================================
# BLOCK 6: EVALUATE ON VALIDATION SET
# ============================================================================

print("="*70)
print("EVALUATING ON VALIDATION SET")
print("="*70)

# Load best model from training
best_model_path = '/kaggle/working/shoulder_arm_yolo/run1_100epochs/weights/best.pt'
model = YOLO(best_model_path)

# Run validation
val_results = model.val(data=yaml_path, split='val')

print("\n📊 VALIDATION RESULTS:")
print("-"*50)
print(f"   mAP50 (IoU=0.50): {val_results.box.map50:.2f}%")
print(f"   mAP50-95 (IoU=0.50:0.95): {val_results.box.map:.2f}%")
print(f"   Precision: {val_results.box.mp:.2f}%")
print(f"   Recall: {val_results.box.mr:.2f}%")
print("-"*50)
print("="*70)

# **BLOCK 8: VISUALIZE TRAINING PROGRESS**

In [ ]:
# ============================================================================
# BLOCK 8: VISUALIZE TRAINING PROGRESS
# ============================================================================

print("="*70)
print("VISUALIZING TRAINING METRICS")
print("="*70)

# Path to results
results_dir = '/kaggle/working/shoulder_arm_yolo/run1_100epochs'

# Plot training metrics if available
if os.path.exists(f"{results_dir}/results.png"):
    from IPython.display import Image as IPImage
    display(IPImage(f"{results_dir}/results.png"))
    print("✅ Training metrics plot loaded")
else:
    print("⚠️ results.png not found yet")

# Print final metrics summary
print("\n📊 FINAL METRICS SUMMARY:")
print("="*50)
print(f"Best mAP50 (Validation): {val_results.box.map50:.2f}%")
print(f"Best mAP50 (Test): {test_results.box.map50:.2f}%")
print(f"Precision: {test_results.box.mp:.2f}%")
print(f"Recall: {test_results.box.mr:.2f}%")
print("="*50)

# **BLOCK 9: SAVE BEST MODEL FOR DOWNLOAD**

In [ ]:
# ============================================================================
# BLOCK 9: SAVE BEST MODEL FOR DOWNLOAD
# ============================================================================

print("="*70)
print("SAVING BEST MODEL FOR DOWNLOAD")
print("="*70)

# Copy best model to working directory
best_model_source = '/kaggle/working/shoulder_arm_yolo/run1_100epochs/weights/best.pt'
best_model_dest = '/kaggle/working/yolo_best_model.pt'
last_model_dest = '/kaggle/working/yolo_last_model.pt'

if os.path.exists(best_model_source):
    shutil.copy(best_model_source, best_model_dest)
    print(f"✅ Best model saved to: {best_model_dest}")
    
    # Also copy last model
    last_model_source = '/kaggle/working/shoulder_arm_yolo/run1_100epochs/weights/last.pt'
    if os.path.exists(last_model_source):
        shutil.copy(last_model_source, last_model_dest)
        print(f"✅ Last model saved to: {last_model_dest}")
else:
    print(f"❌ Model not found at: {best_model_source}")

# Create downloadable link
from IPython.display import FileLink, display

print("\n📥 Download links:")
if os.path.exists(best_model_dest):
    display(FileLink(best_model_dest))
if os.path.exists(last_model_dest):
    display(FileLink(last_model_dest))

# Save results to JSON
import json
results_summary = {
    'train_images': train_count,
    'val_images': val_count,
    'test_images': test_count,
    'epochs': 100,
    'val_mAP50': float(val_results.box.map50),
    'val_mAP50_95': float(val_results.box.map),
    'test_mAP50': float(test_results.box.map50),
    'test_mAP50_95': float(test_results.box.map),
    'precision': float(test_results.box.mp),
    'recall': float(test_results.box.mr)
}

with open('/kaggle/working/training_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("\n✅ Results saved to: /kaggle/working/training_results.json")
print("="*70)